# SingleTaskMultiFidelityGP

`SingleTaskMultiFidelityGP` は、低コスト・低精度の評価と高コスト・高精度の評価を同時に利用するための Gaussian Process です。

この Notebook では、1 次元の設計変数 `x` と fidelity `s` を使い、低 fidelity と高 fidelity のデータを統合して target fidelity `s=1.0` を予測します。

## 1. このモデルを使う場面

例えば次のように、同じ設計に対して複数の評価精度・コストが存在する場合に向いています。

- 粗いシミュレーション / 高精度シミュレーション
- 小規模試験 / 実機試験
- 簡易測定 / 精密測定
- 少ない反復回数 / 十分な反復回数

fidelity 間に相関があることが重要です。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import SingleTaskMultiFidelityGP

torch.manual_seed(0)
dtype = torch.double

## 2. Synthetic example data

入力の 0 列目を設計変数 `x`、1 列目を fidelity `s` とします。

ここでは `s=1.0` を target fidelity とし、低 fidelity ほど少しバイアスが入る関数を人工的に作ります。

In [ ]:
def objective(x: torch.Tensor, fidelity: torch.Tensor) -> torch.Tensor:
    true_value = torch.sin(2 * torch.pi * x) + 0.25 * torch.cos(4 * torch.pi * x)
    bias = (1.0 - fidelity) * (0.8 * (x - 0.5) + 0.25)
    return true_value + bias

x_low = torch.linspace(0.0, 1.0, 12, dtype=dtype).unsqueeze(-1)
s_low = torch.full_like(x_low, 0.25)

x_mid = torch.linspace(0.05, 0.95, 8, dtype=dtype).unsqueeze(-1)
s_mid = torch.full_like(x_mid, 0.60)

x_high = torch.tensor([[0.10], [0.30], [0.55], [0.75], [0.92]], dtype=dtype)
s_high = torch.ones_like(x_high)

train_X = torch.cat(
    [
        torch.cat([x_low, s_low], dim=-1),
        torch.cat([x_mid, s_mid], dim=-1),
        torch.cat([x_high, s_high], dim=-1),
    ],
    dim=0,
)

noise = 0.03 * torch.randn(train_X.shape[0], 1, dtype=dtype)
train_Y = objective(train_X[:, :1], train_X[:, 1:2]) + noise

train_X.shape, train_Y.shape

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for fidelity in [0.25, 0.60, 1.00]:
    mask = torch.isclose(
        train_X[:, 1],
        torch.tensor(fidelity, dtype=dtype),
    )
    ax.scatter(
        train_X[mask, 0],
        train_Y[mask, 0],
        label=f"observations: fidelity={fidelity:.2f}",
    )

x_plot = torch.linspace(0, 1, 300, dtype=dtype).unsqueeze(-1)
y_true = objective(x_plot, torch.ones_like(x_plot))

ax.plot(x_plot.squeeze(), y_true.squeeze(), label="target function: fidelity=1.0")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## 3. Model construction

fidelity は入力の 1 列目なので `data_fidelities=[1]` と指定します。

robotorchan の wrapper は BoTorch の予測挙動を維持しつつ、`raw_*` と `make_mll()` を追加します。

In [ ]:
model = SingleTaskMultiFidelityGP(
    train_X=train_X,
    train_Y=train_Y,
    data_fidelities=[1],
)

model

## 4. robotorchan common API

In [ ]:
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data_names:", model.raw_data_names)
print("supports_mll:", model.supports_mll)

## 5. Model fitting

Exact GP なので、`make_mll()` で marginal log likelihood を作成し、BoTorch の `fit_gpytorch_mll` で学習できます。

In [ ]:
mll = model.make_mll()
fit_gpytorch_mll(mll)

## 6. Posterior prediction at multiple fidelities

同じ `x` に対し fidelity を変えて posterior を比較します。

In [ ]:
model.eval()

x_test = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)

predictions = {}
with torch.no_grad():
    for fidelity in [0.25, 0.60, 1.00]:
        fidelity_column = torch.full_like(x_test, fidelity)
        test_X = torch.cat([x_test, fidelity_column], dim=-1)
        posterior = model.posterior(test_X)
        mean = posterior.mean.squeeze(-1)
        std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
        predictions[fidelity] = (mean, std)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for fidelity, (mean, _) in predictions.items():
    ax.plot(
        x_test.squeeze(),
        mean,
        label=f"posterior mean: fidelity={fidelity:.2f}",
    )

target = objective(x_test, torch.ones_like(x_test))
ax.plot(x_test.squeeze(), target.squeeze(), linestyle="--", label="true target fidelity")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## 7. Target-fidelity uncertainty

実際の最適化では、最終的に興味のある fidelity（ここでは `s=1.0`）での予測が重要です。

In [ ]:
mean, std = predictions[1.00]
lower = mean - 1.96 * std
upper = mean + 1.96 * std

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_test.squeeze(), mean, label="posterior mean at fidelity=1.0")
ax.fill_between(
    x_test.squeeze(),
    lower,
    upper,
    alpha=0.2,
    label="95% interval",
)

target = objective(x_test, torch.ones_like(x_test)).squeeze(-1)
ax.plot(x_test.squeeze(), target, linestyle="--", label="true target")

high_mask = torch.isclose(
    train_X[:, 1],
    torch.tensor(1.0, dtype=dtype),
)
ax.scatter(
    train_X[high_mask, 0],
    train_Y[high_mask, 0],
    label="high-fidelity observations",
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## 8. Simple target-fidelity candidate search

ここでは acquisition function の multi-fidelity 化までは行わず、posterior mean を使って target fidelity 上の候補を確認します。

本格的な Multi-Fidelity BO では、評価コストを考慮した acquisition function や fidelity 選択を組み合わせます。

In [ ]:
candidate_index = predictions[1.00][0].argmax()
candidate_x = x_test[candidate_index]
candidate_X = torch.cat(
    [candidate_x, torch.ones_like(candidate_x)],
    dim=-1,
)

print("candidate at target fidelity:", candidate_X)
print(
    "posterior mean:",
    predictions[1.00][0][candidate_index].item(),
)

## 9. When to use / when not to use

### Use

- fidelity の段階が明確に存在する
- 低 fidelity が高 fidelity と相関する
- 高 fidelity の評価コストが高い
- 低 fidelity データを有効活用したい

### Consider another model

- fidelity が単なる通常の説明変数にすぎない
- fidelity 間にほとんど相関がない
- 評価コストがほぼ同じ
- fidelity ではなく「装置 A / 装置 B」のような離散タスクとして扱う方が自然

その場合は `SingleTaskGP` や `MultiTaskGP` も候補になります。